# 04 — Feature Engineering
**FlightIQ — AI Travel Price Intelligence**

---
Builds the sklearn preprocessing pipeline. Documents every feature decision and leakage check.


In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
sys.path.insert(0, os.path.abspath('../src'))
from feature_engineering import (
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES, ALL_FEATURES, TARGET,
    build_preprocessor, prepare_data, load_data
)
df = load_data()
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

Dataset: 93,083 rows x 27 columns


## 1. Feature Selection & Leakage Audit

In [2]:
print('NUMERICAL FEATURES:')
for f in NUMERICAL_FEATURES:
    print(f'  {f:<35} missing={df[f].isna().sum()}')
print(f'\nCATEGORICAL FEATURES:')
for f in CATEGORICAL_FEATURES:
    print(f'  {f:<35} nunique={df[f].nunique()}  missing={df[f].isna().sum()}')
print(f'\nTOTAL RAW FEATURES: {len(ALL_FEATURES)}')
print(f'TARGET: {TARGET}')


NUMERICAL FEATURES:
  Distance_km                         missing=0
  Duration_Minutes                    missing=0
  Total_Stops_Numeric                 missing=0
  Days_Before_Departure               missing=0
  Departure_Time_Minutes              missing=0
  Arrival_Time_Minutes                missing=0
  Departure_Month                     missing=0
  Departure_DayOfWeek_Num             missing=0
  Passenger_Count                     missing=0

CATEGORICAL FEATURES:
  Airline                             nunique=13  missing=0
  Source                              nunique=18  missing=0
  Destination                         nunique=18  missing=0
  Travel_Class                        nunique=4  missing=0
  Season                              nunique=4  missing=0
  Weekday                             nunique=7  missing=0
  Aircraft_Type                       nunique=8  missing=0
  Booking_Channel                     nunique=5  missing=0

TOTAL RAW FEATURES: 17
TARGET: Price


In [3]:
print('EXCLUDED COLUMNS & REASONS:')
excluded = {
    'Flight_ID':'Identifier only — no predictive signal',
    'Price':'TARGET — must never be a feature (leakage)',
    'Duration':'Raw string — Duration_Minutes is clean version',
    'Total_Stops':'Raw string — Total_Stops_Numeric is clean version',
    'Departure_Date':'Raw string with ~5K missing — Month/DayOfWeek extracted',
    'Departure_Time':'Raw string with ~5K missing — Departure_Time_Minutes extracted',
    'Arrival_Time':'Raw string with ~4.5K missing — Arrival_Time_Minutes extracted',
    'Departure_Date_Parsed':'Datetime with missing — numeric features already extracted',
    'Departure_DayOfYear':'High cardinality (365), collinear with Month — excluded',
    'Weekday_Num':'Duplicate of Departure_DayOfWeek_Num — excluded',
}
for col, reason in excluded.items():
    print(f'  {col:<30}: {reason}')


EXCLUDED COLUMNS & REASONS:
  Flight_ID                     : Identifier only — no predictive signal
  Price                         : TARGET — must never be a feature (leakage)
  Duration                      : Raw string — Duration_Minutes is clean version
  Total_Stops                   : Raw string — Total_Stops_Numeric is clean version
  Departure_Date                : Raw string with ~5K missing — Month/DayOfWeek extracted
  Departure_Time                : Raw string with ~5K missing — Departure_Time_Minutes extracted
  Arrival_Time                  : Raw string with ~4.5K missing — Arrival_Time_Minutes extracted
  Departure_Date_Parsed         : Datetime with missing — numeric features already extracted
  Departure_DayOfYear           : High cardinality (365), collinear with Month — excluded
  Weekday_Num                   : Duplicate of Departure_DayOfWeek_Num — excluded


## 2. Build & Inspect Preprocessing Pipeline

In [4]:
from sklearn.model_selection import train_test_split
X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Train: {X_train.shape[0]:,} rows')
print(f'Test : {X_test.shape[0]:,} rows')
print(f'\nTarget range — Train: ₹{y_train.min():,.0f} – ₹{y_train.max():,.0f}')
print(f'Target range — Test : ₹{y_test.min():,.0f} – ₹{y_test.max():,.0f}')


Train: 74,466 rows
Test : 18,617 rows

Target range — Train: ₹152 – ₹999,306
Target range — Test : ₹157 – ₹989,081


In [5]:
preprocessor = build_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
print(f'Raw features     : {len(ALL_FEATURES)}')
print(f'Encoded features : {X_train_proc.shape[1]}')
print(f'  (OHE expanded {len(CATEGORICAL_FEATURES)} categoricals)')
print(f'Train matrix : {X_train_proc.shape}')
print(f'Test matrix  : {X_test_proc.shape}')
print(f'NaN in train : {np.isnan(X_train_proc).sum()}')
print(f'NaN in test  : {np.isnan(X_test_proc).sum()}')


Raw features     : 17
Encoded features : 86
  (OHE expanded 8 categoricals)
Train matrix : (74466, 86)
Test matrix  : (18617, 86)
NaN in train : 0
NaN in test  : 0


## 3. Feature Names After Encoding

In [6]:
from feature_engineering import get_feature_names
feature_names = get_feature_names(preprocessor)
print(f'Total encoded feature names: {len(feature_names)}')
print('\nNumerical (unchanged):')
for n in NUMERICAL_FEATURES:
    print(f'  {n}')
print('\nCategorical (OHE expanded):')
for n in feature_names[len(NUMERICAL_FEATURES):]:
    print(f'  {n}')


Total encoded feature names: 86

Numerical (unchanged):
  Distance_km
  Duration_Minutes
  Total_Stops_Numeric
  Days_Before_Departure
  Departure_Time_Minutes
  Arrival_Time_Minutes
  Departure_Month
  Departure_DayOfWeek_Num
  Passenger_Count

Categorical (OHE expanded):
  Airline_Air India
  Airline_Airasia India
  Airline_British Airways
  Airline_Emirates
  Airline_Etihad Airways
  Airline_Gofirst
  Airline_Indigo
  Airline_Lufthansa
  Airline_Qatar Airways
  Airline_Singapore Airlines
  Airline_Spicejet
  Airline_Thai Airways
  Airline_Vistara
  Source_Ahmedabad
  Source_Bangalore
  Source_Bangkok
  Source_Chennai
  Source_Delhi
  Source_Doha
  Source_Dubai
  Source_Frankfurt
  Source_Goa
  Source_Hyderabad
  Source_Jaipur
  Source_Kolkata
  Source_London
  Source_Mumbai
  Source_New York
  Source_Pune
  Source_Singapore
  Source_Sydney
  Destination_Ahmedabad
  Destination_Bangalore
  Destination_Bangkok
  Destination_Chennai
  Destination_Delhi
  Destination_Doha
  Destination_

## 4. Summary

In [7]:
print('='*55)
print('FEATURE ENGINEERING SUMMARY')
print('='*55)
print(f'Raw input features : {len(ALL_FEATURES)}')
print(f'Encoded features   : {X_train_proc.shape[1]}')
print(f'Train samples      : {X_train_proc.shape[0]:,}')
print(f'Test samples       : {X_test_proc.shape[0]:,}')
print(f'No NaNs in matrices: {np.isnan(X_train_proc).sum() == 0}')
print(f'Leakage check      : PASSED (Price not in features)')
print('='*55)


FEATURE ENGINEERING SUMMARY
Raw input features : 17
Encoded features   : 86
Train samples      : 74,466
Test samples       : 18,617
No NaNs in matrices: True
Leakage check      : PASSED (Price not in features)
